### Conversation Q&A Chatbot

In [ ]:
#Load API keys from env file and load the model
import os 
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
groq_api_key = os.environ.get("GROQ_TOKEN")
llm = ChatGroq(groq_api_key=groq_api_key, model="openai/gpt-oss-120b")

In [ ]:
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
#bring the hugging face embedding model
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


In [ ]:
from langchain_chroma import Chroma #for vector store
from langchain_community.document_loaders import WebBaseLoader #for web page loading
from langchain_core.prompts import ChatPromptTemplate #for prompt templates
from langchain_text_splitters import RecursiveCharacterTextSplitter #for text splitting
from langchain_core.runnables import RunnableMap, RunnablePassthrough #for running the chain
#from langchain.chains import create_retrieval_chain #for retrieval chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain #for combining documents

In [ ]:
#Load specific data data from web page 
import bs4
loader = WebBaseLoader(
        web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
        bs_kwargs=dict(
            parse_only=bs4.SoupStrainer(
                class_=("post-content", "post-title", "post-header")
            )
        ),
    )

docs = loader.load()
docs

In [ ]:
#sploit the documents into chunks and store it in a vector database
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(docs)
vectorstordb=Chroma.from_documents(documents,embedding_model)
retriever=vectorstordb.as_retriever()
retriever 

In [ ]:
docs

In [ ]:
#prompt template
system_prompt = (
    "You are a helpful assistant."
    "Use the following pieces of retrived context to answer the question."
    "If you don't know the answer, just say that you don't know."
    "Use three sentences maximum and keep the answer concise."
    "\n \n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
   [
    ("system", system_prompt),
    ("user", "{input}")
   ]
   )

   #stuff the document into single string
def combine_documents(docs):
    return "\n\n".join([doc.page_content for doc in docs])
document_chain = RunnableMap(
    {
        "context": retriever | combine_documents,
       "input": RunnablePassthrough()
    }
    ) | prompt|llm

In [ ]:
response = document_chain.invoke("What is self-reflection?")
response

In [ ]:
curl https://api.groq.com/openai/v1/models \
-H "Authorization: Bearer $grow_api_key"